# Homework 4 - Solving Systems of Equations, Gaussian Elimination, and Radiative Transfer (10 pt)

In many areas of astrophysics, it is crucial to understand the propigation of light. An  important discipline of astrophysics is *radiative transfer*, which calculates the intensity of light as it goes through material. Generally to do this calculation, one has to calculate the emission and absorbtion of many light paths, which is often done with matrix algebra. In this homework exercise, we will explore some of the methods of how this calculation is done, starting off with a one-dimensional calculation.

To accept this assignment, click the link below and follow the steps:\
https://classroom50.org/PsuAstro410/410astro26/assignments/homework-4/accept 

This homework assignment was based a problem set originally designed by Shane Davis.

In [ ]:
# Useful things to import and load

%matplotlib inline
import numpy as np
import astropy.constants as cons
import scipy.sparse as sparse
import matplotlib.pyplot as plt

# Theory of Radiation Transfer

In astrophysics, we often want encounter problems where we need to solve for the steady-state structure of an object, like a star or calculated the run of a physical variable as a function of position.  Let's consider the example of radiation transfer through an atmosphere.  The steady state radiation transfer equation for the specific intensity $I_\nu$ in the presence of isotropic scattering is
\begin{equation}
\mu\frac{\partial I_\nu}{\partial z} = \eta_\nu - (\alpha_\nu +\sigma_\nu) I_\nu + \sigma_\nu J_\nu,
\end{equation}
where $\mu$ is the cosine of the angle the ray makes with the vertical, $z$ is the height in the atmosphere, $\eta_\nu$ is the emissivity, $\alpha_\nu$ and $\sigma_\nu$ the extinction coefficients associated with absorption and scattering, and $J_\nu =(4\pi)^{-1} \int I_\nu d\Omega$ is the angle averaged intensity at frequency $\nu$.

Define variables 
$$\epsilon_\nu = \sigma_\nu/(\sigma_\nu+\alpha_\nu)$$
and the optical depth 
$$d \tau_\nu = -(\alpha_\nu+\sigma_\nu) dz.$$ 
Also, use Kirchoff's law to relate $\eta_\nu = B_\nu \alpha_\nu$, where
\begin{equation}
B_\nu=\frac{2 h}{c^2}\frac{\nu^3}{\exp(h\nu/k_B T) -1}.
\end{equation}
Then the transfer equation becomes
\begin{equation}
\mu\frac{\partial I_\nu}{\partial \tau_\nu} = I_\nu - \epsilon_\nu B_\nu -(1-\epsilon_\nu)J_\nu.
\end{equation}

In general, the transfer equation is solve (analytically) for all rays at different angles to the vertical or some number of discrete values. 

In this homework assignment, we will use what is called the two stream approximation to calculate the structure of an atmosphere similar to a massive star, where one assumes a light ray only travels upwards and downwards, corresponding to values of $\mu = \pm 1/\sqrt{3}$ so that $I^+_\nu = I_\nu(\mu=+1/\sqrt{3})$ and $I^-_\nu = I_\nu(\mu=-1/\sqrt{3})$. Hence, the transfer equation becomes
\begin{equation}
\pm\frac{1}{\sqrt{3}}\frac{\partial I^\pm_\nu}{\partial \tau_\nu} = I^\pm_\nu - \epsilon_\nu B_\nu -(1-\epsilon_\nu)J_\nu.
\end{equation}
This approximation gives that
$$J_\nu = \frac{1}{2}(I^+_\nu + I^-_\nu),$$
and a different "moment"
$$
H_\nu = (4\pi)^{-1} \int I \cos \theta d \Omega =  \frac{1}{2\sqrt{3}} (I^+_\nu - I^-_\nu).
$$
we can add or subtract the equations for $I^\pm_\nu$ to obtain
\begin{equation}
\frac{\partial H_\nu}{\partial \tau_\nu} = \epsilon_\nu (J_\nu - B_\nu),
\end{equation}
and
\begin{equation}
\frac{\partial J_\nu}{\partial \tau_\nu} = 3 H_\nu.
\end{equation}
Taking the derivative of the second equation with respect to $\tau_\nu$ and inserting the first equation yields
\begin{equation}
\frac{\partial^2 J_\nu}{\partial \tau_\nu^2} = 3\epsilon_\nu (J_\nu - B_\nu). \qquad (1)
\end{equation}

**Atmosphere top:** We assume no incoming radiation so that $I^-_\nu =0$. This means that
\begin{equation}
J_\nu = \sqrt{3} H_\nu = \frac{1}{\sqrt{3}} \frac{\partial J_\nu}{\partial \tau_\nu}. \qquad (2)
\end{equation}

**Atmosphere bottom:** We assume local thermodynamic equilibrium, so that the emissivity is equal to the blackbody emission at the atmosphere base, so  
$$
J_\nu = B_\nu. \qquad (3)
$$

We now have everything we need to solve the transfer equation for $J_\nu$ assuming we know $B_\nu$ as a function of $\tau_\nu$.

# Problem 1 - Scattering and absorbtion coefficients (3 pt)

We want to write functions for computing $B_\nu$, $\alpha_\nu$, and $\sigma_\nu$ given density $\rho(z)$ and temperature $T(z)$.  Assume that $\alpha_\nu = \kappa^{ff}_\nu \rho$ and $\sigma_\nu = \kappa_{\rm es} \rho$.  Our atmosphere will consist of ionized H, so that
\begin{equation}
\sigma_\nu = \frac{\sigma_{\rm T}}{m_p} \rho
\end{equation}
We approximate the absorbtion being completely free-free, so that
\begin{equation}
\alpha_\nu \approx \frac{4 e^6}{3 m_p^2 m_e h c} \left(\frac{2\pi}{3 k_B m_e} \right)^{1/2} \rho^2 T^{-1/2} \nu^{-3} \left[1- \exp(- h\nu/k_B T)\right].
\end{equation}
Here $e$ is the electron charge, $m_p$ is the proton mass, $k_B$ is Boltzmann's constant, $h$ is Planck's constant, and $c$ is the speed of light. Below, I have loaded these constants using the `astropy` module:

In [ ]:
# All quantities are in grams-centemeters-seconds units

h = cons.h.cgs.value             # Plank's constant
k_B = cons.k_B.cgs.value         # Boltzman constant
m_p =  cons.m_p.cgs.value        # Proton mass
m_e = cons.m_e.cgs.value          # Electron mass
ec = cons.e.esu.value            # Electron charge
c = cons.c.cgs.value             # Speed of light
sigma_T = cons.sigma_T.cgs.value # Thompson cross-section

## Problem 1(a) - Write absorbtion coefficient, scattering coefficient, and planck function (2 pt)

For this problem, create functions that calculate
* `absorption_extinction()`, for $\alpha_\nu$, which takes arguments $\rho$, $T$, and $\nu$
* `scattering_extinction()`, for $\sigma_\nu$, which takes arguments $\rho$
* `planck()`, for $B_\nu$, which takes arguments $T$ and $\nu$

The arguments $\rho$ and $T$ will be NumPy arrays.  $\nu$ will be a float (a single specific frequency) in the matrix equation we derive below. 

In [ ]:
## Answer here


## Problem 1(b) - Validate functions with comparison case (1 pt)

For a density of $\rho = 10^{-5} \ {\rm g}/{\rm cm}^3$, temperature of $T = 10^4 \ {\rm K}$, and frequency values $\nu$ between $10^{12}$ and $10^{15.5}$ Hz, I have calculated the "correct" values for these coefficients, in the arrays below for the frequency values indicated in `nu_func_test`, with all quantities in cgs units.

In [ ]:
nu_func_test = np.array([1.00000000e+12, 2.44843675e+12, 5.99484250e+12, 1.46779927e+13,
       3.59381366e+13, 8.79922544e+13, 2.15443469e+14, 5.27499706e+14,
       1.29154967e+15, 3.16227766e+15])
abs_ext_func_test = np.array([6.31885235e+07, 1.05039594e+07, 1.73736823e+06, 2.83883292e+05,
       4.50458037e+04, 6.67287311e+03, 8.50485169e+02, 8.27655005e+01,
       6.11350436e+00, 4.17356397e-01])
scat_ext_func_test = 3.977263862130838e-05
planck_func_test = np.array([3.06499177e-09, 1.83103030e-08, 1.08834095e-07, 6.38879358e-07,
       3.63573445e-06, 1.91178048e-05, 8.13653537e-05, 1.87000535e-04,
       6.46999976e-05, 1.19552186e-07])

Compare the outputs of your functions, and verify that your functions agree with these values. For your own array, plot your function as having 100 values of $\nu$ spaced logarithmically between $10^{12}$ and $10^{15.5}$ Hz, and plot your result as a blue line. Plot the test "correct" values as black points. For the scattering cross-section, print your result as `"scattering = "`, and verify that the quantity displayed in cgs units is similar.

In [ ]:
## Answer here


# Problem 2 - Solving the Matrix Equation (4 pt)

We want to solve the transfer equation to compute $J_\nu^{(i)}$ on a non-uniform grid of $\tau_\nu$ running from $\tau_\nu^{(0)}$ to $\tau_\nu^{(N-1)}$. Take the first step towards this by creating a matrix equation of the form
\begin{equation}
{\sf A} \cdot \mathbf{J} = \mathbf{b},
\end{equation}
where the vector $\mathbf{J}$ is the value a $J_\nu^{(i)}$ at the $N$ points.

## Problem 2(a) - Creating the Matrix Equation (3 pt)

One way of doing this is to rewrite equation (1) as:
\begin{equation}
-\frac{1}{3 \epsilon_\nu^{(i)}}\left(\frac{\partial^2 J_\nu}{\partial \tau_\nu^2}\right)^{(i)} + J_\nu^{(i)} = B_\nu^{(i)}.
\end{equation}

Unlike the derivatives we dealt with before, the variation in $\tau$ from $\tau^{(0)}$ to $\tau^{(N-1)}$ might not be uniform: e.g. the spacing $\tau_\nu^{(i+1)} - \tau_\nu^{(i)}$ might not give the same $\Delta \tau_\nu$ for all $i$. For this reason, we need to be careful with our discretization of the second derivative. 

For our problem, lets approximate the first derivative at a mid-way index $i+1/2$ to be
\begin{equation}
\left(\frac{\partial J_\nu}{\partial \tau_\nu}\right)^{(i+1/2)} \approx \frac{J_\nu^{(i+1)}-J_\nu^{(i)}}{\tau_\nu^{(i+1)}-\tau_\nu^{(i)}}.
\end{equation}
Here, the index notation means that integers $(i)$ are evaluated on the grid of points $z$ and 1/2 integers $(i \pm 1/2)$ are evaluated at midpoints.  Similarly,
\begin{equation}
\left(\frac{\partial J_\nu}{\partial \tau_\nu}\right)^{(i-1/2)} \approx \frac{J_\nu^{(i)}-J_\nu^{(i-1)}}{\tau_\nu^{(i)}-\tau_\nu^{(i-1)}}.
\end{equation}
We can then approximate
\begin{equation}
\left(\frac{\partial^2 J_\nu}{\partial \tau_\nu^2}\right)^{(i)} \approx \frac{1}{\tau_\nu^{(i+1/2)}-\tau_\nu^{(i-1/2)}} \left[ \left(\frac{\partial J_\nu}{\partial \tau_\nu}\right)^{(i+1/2)} - \left(\frac{\partial J_\nu}{\partial \tau_\nu}\right)^{(i-1/2)} \right] \qquad (5)
\end{equation}
To estimate variables at the half-integer points, we can take an average e.g. $\tau_\nu^{(i+1/2)} = (\tau_\nu^{(i+1)}+\tau_\nu^{(i)})/2$ and $\tau_\nu^{(i-1/2)} = (\tau_\nu^{(i)}+\tau_\nu^{(i-1)})/2$.

For the boundary condition at the top of the atmosphere ($i=0$), we can approximate the derivative as a forward difference
\begin{equation}
J_\nu^{(0)} \approx \frac{1}{\sqrt{3}}\frac{J_\nu^{(1)}-J_\nu^{(0)}}{\tau_\nu^{(1)}-\tau_\nu^{(0)}}, \qquad (6)
\end{equation}
and at the base ($i=N-1$),
\begin{equation}
J_\nu^{(N-1)}=B_\nu^{(N-1)}. \qquad (7)
\end{equation}

In the cell below, write a function that returns the matrix ${\sf A}$ and vector $\mathbf{b}$. This requires working out the matrix elements that, for a given system of equations at the index $i$, one multiplies $J_\nu^{(i+1)}$, $J_\nu^{(i)}$, and $J_\nu^{(i-1)}$ for each $i = 1, ..., N-2$, to get a linear system of equations, with the boundary conditions giving matrix elements for $i = 0$ and $i = N-1$. 

The vector $\mathbf{b}$ is just 
$$
b_i = B_\nu^{(i)} \qquad (8)
$$
for all $i$ except $i=0$, where eq. (7) gives
$$
b_0 = 0 \qquad (9)
$$

Call your function `transfer_matrices()`, which takes the following arguments:
* `tau`: an $n$ element array of optical depths $\tau_\nu$
* `B_nu`: an $n$ element array of the Planck function $B_\nu$
* `epsilon_nu`: an $n$ element array of the opacity fraction $\epsilon_\nu$

and returns an $N\times N$ matrix ${\sf A}$ and $N$ element vector $\mathbf{b}$

In [ ]:
## Answer here


## Problem 2(b) - Solving the matrix equation for a test case (1 pt)

To verify that numerical algorithms are calculating the correct solution, one often tests the validity of the algorithm with an example that is known analytically. If $B_\nu$ and $\epsilon_\nu$ are both constant, one can calculate an analytic solution to equation (1):
$$
J_\nu = B_\nu \left[1 -\frac{\exp\left(-\sqrt{3 \epsilon_\nu} \tau_\nu\right)}{1+\sqrt{\epsilon_\nu}} \right]
$$

To solve this matrix equation, we will use the numerical algorithm that we convered in class:

In [ ]:
def print_Ab(A, b):
    """printout the matrix A and vector b in a pretty fashion."""

    N = len(b)

    space = 8*" "
    top_str = "⎧" + N*" {:>7.03f} " + "⎫" + space + "⎧" + " {:6.3f} " + "⎫"
    sid_str = "⎪" + N*" {:>7.03f} " + "⎪" + space + "⎪" + " {:6.3f} " + "⎪"
    bot_str = "⎩" + N*" {:>7.03f} " + "⎭" + space + "⎩" + " {:6.3f} " + "⎭"

    for i in range(N):
        if i == 0:
            pstr = top_str
        elif i == N-1:
            pstr = bot_str
        else:
            pstr = sid_str
        out = tuple(A[i, :]) + (b[i],)
        print(pstr.format(*out))
    print(" ")

def gauss_elim(A, b, *, quiet=False, pivot=True):
    """ perform gaussian elimination with pivoting, solving A x = b.

        A is an NxN matrix, x and b are an N-element vectors.  Note: A
        and b are changed upon exit to be in upper triangular (row
        echelon) form """

    assert b.ndim == 1, "ERROR: b should be a vector"

    N = len(b)
    assert A.shape == (N, N), "ERROR: A should be square with each dim of same length as b"

    x = np.zeros((N), dtype=A.dtype)

    if not quiet:
        print_Ab(A, b)

    # main loop over rows
    for k in range(N):

        if not quiet:
            print(f"working on row {k}")

        if pivot:
            # find the pivot row based on the size of column k -- only consider
            # the rows >= k (then add k to the index so it is 0-based)
            row_max = np.argmax(np.abs(A[k:, k])) + k

            if row_max != k:
                # swap the row with the largest element in the current column
                # with the current row (pivot) -- do this with b too!
                A[[k, row_max], :] = A[[row_max, k], :]
                b[[k, row_max]] = b[[row_max, k]]
                if not quiet:
                    print("pivoted, updated system:")
                    print_Ab(A, b)

        # do the forward-elimination for all rows below the current
        for i in range(k+1, N):
            coeff = A[i, k] / A[k, k]

            for j in range(k+1, N):
                A[i, j] += -A[k, j] * coeff

            A[i, k] = 0.0
            b[i] += -coeff * b[k]
            
            # check if the row is all zeros -- singular
            if np.abs(A[i, :]).max() == 0:
                if not quiet:
                    print("singular")
                    print_Ab(A, b)
                raise ValueError("matrix is singular")

        if not quiet:
            print_Ab(A, b)

    # back-substitution

    # last solution is easy
    x[N-1] = b[N-1] / A[N-1, N-1]

    for i in reversed(range(N-1)):
        bsum = b[i]
        for j in range(i+1, N):
            bsum += -A[i, j] * x[j]
        x[i] = bsum / A[i, i]

    return x

Test the numerical algorithm, by creating a logarithmic array of 100 $\tau$ values from $10^{-3}$ to $10^3$, setting $\epsilon_\nu = 0.5$, taking $\nu = 10^{15}$ Hz and $T = 10^5$ K for $B_\nu$. Compare the numerically-computed solution to the matrix equation to the analytic solution. Make your numerical solution solid blue, and your analytic solution dotted black. If your matrix solver is working, the two should agree pretty well.

In [ ]:
## Answer here


# Problem 3 - Calculate Solution for Model Atmosphere (3 pt)

We will calculate the radiative transfer for an atmosphere that is similar to that of an O-star. For such atmospheres, the temperature is roughly isothermal, so at all depths $z$, the temperature is a constant
$$
T(z) \approx 10^5 \ {\rm K} = \text{constant},
$$
with a constant pressure scale-height $H = (d \ln P/d r)^{-1} = c_s^2/g$ of
$$
H = \frac{z_{\rm max} - z_{\rm min}}{\ln(\tau_{\rm max}/\tau_{\rm min})}.
$$
The density tapers off exponentially, as
$$
\rho(z) = \frac{\tau_{\rm min}}{H \kappa_{\rm es}} \exp \left( \frac{z - z_{\rm min}}{H} \right),
$$
where $\kappa_{\rm es}$ is the opacity due to electron scattering. For an O-star, we will roughly take $z_{\rm min} = 0$, $z_{\rm max} = 10^{10}$ cm, $\tau_{\rm min} = 10^{-3}$, $\tau_{\rm max} = 10^3$, and $\kappa_{\rm es} = 0.3 \ {\rm cm}^2/{\rm g}$. Notice that because we are plotting with depth, the density grows with $z$.

For a frequency of $\nu = 10^{15}$ Hz, we will calculate the solution to the radiative transfer equation with the following steps:

## Problem 3(a) - Initialize atmosphere, and calculate the emissivity, absorbtion, and source function (1 pt)

After initializing this atmosphere, use the functions in Problem 1 to calculate $\alpha_\nu$, $\sigma_\nu$, $\epsilon_\nu$, and $B_\nu$. Do this at all depths $z$. Have the grid in $z$ be linear with 200 data points.

In [ ]:
## Answer here


## Problem 3(b) - Calculate the optical depth through the atmosphere (1 pt)

Starting at the optical depth $\tau_\nu^0 = \tau_{\rm min}$, calculate $\tau_\nu$ with depth at every inded $i$, using
$$
\tau_\nu^{(i)} \approx \tau_\nu^{(i-1)} + (\alpha_\nu+\sigma_\nu)^{(i-1/2)} \Delta z^{(i)}.
$$
with 
$$
(\alpha_\nu+\sigma_\nu)^{(i-1/2)} = \frac{1}{2} \left[ \left( \alpha_\nu+\sigma_\nu \right)^{(i)} +  \left( \alpha_\nu+\sigma_\nu \right)^{(i-1)} \right]
$$

In [ ]:
## Answer here


## Problem 3(c) - Determine the intensity with depth, and plot the solution (1 pt)

Using the `gaus_elim` routine above, determine the solution $J_\nu$ to the radiative transfer equation. Initialize the radiative transfer matrix with `transfer_matrices` defined above, and solve. Calculate $J_\nu/B_\nu$ for different optical depth, and calculate $\tau$ on a logarithmic axis, and $J_\nu/B_\nu$ on a linear axis. How does the ratio $J_\nu/B_\nu$ change with depth?

In [ ]:
## Answer here
